# Chatbot with MCP

In [3]:
pip install arxiv

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 3.0 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=1d4efe5e2397b3e09690b68c0ee300ea583aa43055d54bd404538286bdf869c2
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k


In [4]:
pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.2/388.2 kB 15.1 MB/s eta 0:00:00


In [38]:
import arxiv
import json
import os
from typing import List
from dotenv import load_dotenv
import anthropic
from mcp.server.fastmcp import FastMCP

## Tool functions

In [2]:
PAPER_DIR = "papers"

In [39]:
# Initialize FastMCP server
mcp = FastMCP("research")

### First tool

The first tool searches for relevant arXiv papers based on a topic and stores the papers' info in a JSON file (title, authors, summary, paper url and the publication date). The JSON files are organized by topics in the papers directory. The tool does not download the papers.

In [40]:
@mcp.tool()
def search_papers(topic: str, max_results: int = 5) -> List[str]:
    """
    Search for relevant arXiv based on a topic and store their information.

    Args:
        topic (str): The topic to search for.
        max_results (int, optional): The maximum number of results to retrieve. Defaults to 5.

    Returns:
        List of paper IDs found in the search
    """

    # Use arXiv to find the papers
    client = arxiv.Client()

    # Search for the most relevant articles matching the queried topic
    search = arxiv.Search(
        query=topic
        ,max_results=max_results
        ,sort_by=arxiv.SortCriterion.Relevance
    )

    papers = client.results(search)

    # Create directory for this topic
    path = os.path.join(PAPER_DIR, topic.lower().replace(" ", "_"))
    os.makedirs(path, exist_ok=True)

    file_path = os.path.join(path, "papers_info.json")

    # Try to load existing papers info
    try:
      with open(file_path, "r") as json_file:
        papers_info = json.load(json_file)
    except (FileNotFoundError, json.JSONDecodeError):
      papers_info = {}

    # Process each paper and add to papers_info
    paper_ids = []
    for paper in papers:
        paper_ids.append(paper.get_short_id())
        paper_info = {
            "title": paper.title
            ,"authors": [author.name for author in paper.authors]
            ,"summary": paper.summary
            ,"pdf_url": paper.pdf_url
            ,"published": str(paper.published.date())
        }
        papers_info[paper.get_short_id()] = paper_info

    # Save updated papers_info to json file
    with open(file_path, "w") as json_file:
        json.dump(papers_info, json_file, indent=2)

    print(f"Results are saved in :{file_path}")

    return paper_ids



In [4]:
search_papers("computers")

Results are saved in :papers/computers/papers_info.json


['1312.3300v1', '2207.05241v1', '2012.10468v1', '2009.00041v1', '2009.08005v1']

In [5]:
search_papers("LLM user facilities")

Results are saved in :papers/llm_user_facilities/papers_info.json


['2511.00176v1',
 '2211.13450v1',
 '2306.05212v1',
 '2512.10895v1',
 '2407.04039v1']

### Second tool


The second tool looks for information about a specific paper across all topic directories inside the papers directory.

In [41]:
@mcp.tool()
def extract_info(paper_id: str) -> str:
    """
    Search for information about a specific paper across all topic directories.

    Args :
      paper_id : The ID of the paper to look for

    Returns:
      JSON string with paper information if found, error message if not found
    """

    for item in os.listdir(PAPER_DIR):
        item_path = os.path.join(PAPER_DIR, item)
        if os.path.isdir(item_path):
          file_path = os.path.join(item_path, "papers_info.json")
          if os.path.isfile(file_path):
            try:
              with open(file_path, "r") as json_file:
                papers_info = json.load(json_file)
                if paper_id in papers_info:
                  return json.dumps(papers_info[paper_id], indent=2)
            except (FileNotFoundError, json.JSONDecodeError) as e:
              print(f"Error reading {file_path} : {str(e)}")
              continue

    return f"There's no saved information related to paper {paper_id}."

In [7]:
extract_info("2512.10895v1")

'{\n  "title": "LLMs Can Assist with Proposal Selection at Large User Facilities",\n  "authors": [\n    "Lijie Ding",\n    "Janell Thomson",\n    "Jon Taylor",\n    "Changwoo Do"\n  ],\n  "summary": "We explore how large language models (LLMs) can enhance the proposal selection process at large user facilities, offering a scalable, consistent, and cost-effective alternative to traditional human review. Proposal selection depends on assessing the relative strength among submitted proposals; however, traditional human scoring often suffers from weak inter-proposal correlations and is subject to reviewer bias and inconsistency. A pairwise preference-based approach is logically superior, providing a more rigorous and internally consistent basis for ranking, but its quadratic workload makes it impractical for human reviewers. We address this limitation using LLMs. Leveraging the uniquely well-curated proposals and publication records from three beamlines at the Spallation Neutron Source (SN

In [8]:
extract_info('1310.7911v2')

"There's no saved information related to paper 1310.7911v2."

## Tool Schema


Here are the schema of each tool which you will provide to the LLM.

In [9]:
tools = [
    {
        "name": "search_papers",
        "description": "Search for papers on arXiv based on a topic and store their information.",
        "input_schema": {
            "type": "object",
            "properties": {
                "topic": {
                    "type": "string",
                    "description": "The topic to search for"
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum number of results to retrieve",
                    "default": 5
                }
            },
            "required": ["topic"]
        }
    },
    {
        "name": "extract_info",
        "description": "Search for information about a specific paper across all topic directories.",
        "input_schema": {
            "type": "object",
            "properties": {
                "paper_id": {
                    "type": "string",
                    "description": "The ID of the paper to look for"
                }
            },
            "required": ["paper_id"]
        }
    }
]

## Tool Mapping


This code handles tool mapping and execution.

In [27]:
mapping_tool_function = {
    "search_papers": search_papers,
    "extract_info": extract_info
}

def execute_tool(tool_name, tool_args):

    result = mapping_tool_function[tool_name](**tool_args)

    if result is None:
        result = "The operation completed but didn't return any results."

    elif isinstance(result, list):
        result = ', '.join(result)

    elif isinstance(result, dict):
        # Convert dictionaries to formatted JSON strings
        result = json.dumps(result, indent=2)

    else:
        # For any other type, convert using str()
        result = str(result)
    return result

## Chatbot with function calling

The chatbot handles the user's queries one by one, but it does not persist memory across the queries.

In [29]:
max_tokens = 2024
model = "claude-haiku-4-5-20251001"
client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
print(client)

message = client.messages.create(
    model=model,
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "Hello, Claude"}
    ]
)
print(message)



Message(id='msg_01DfqyxfavdK9z4rxHxkrE2v', content=[TextBlock(citations=None, text="Hello! It's nice to meet you. How can I help you today?", type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=10, output_tokens=19, server_tool_use=None, service_tier='standard'))


In [34]:
def process_query(query):
    """
    Processes a user query through an agentic loop with tool calling.
    Continues until LLM returns a text-only response (no tool requests).
    """

    messages = [{'role': 'user', 'content': query}]

    response = client.messages.create(max_tokens=max_tokens
                                      ,model=model
                                      ,tools=tools
                                      ,messages=messages)

    # Agentic loop: continue while LLM requests tools
    should_continue = True
    while should_continue:
        assistant_content = []
        tool_results = []

        # Process each content block in LLM's response
        for content in response.content:
            if content.type == 'text':
                # Text response: display and store
                print(content.text)
                assistant_content.append(content)

            elif content.type == 'tool_use':
                # Tool request: execute and collect results
                assistant_content.append(content)

                tool_id = content.id
                tool_args = content.input
                tool_name = content.name
                print(f"Calling tool {tool_name} with args {tool_args}")

                # Execute tool and format result
                result = execute_tool(tool_name, tool_args)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": tool_id,
                    "content": result
                })

        # If tools were called, send all results back to Claude
        if tool_results:
            messages.append({"role": "assistant", "content": assistant_content})
            messages.append({"role": "user", "content": tool_results})

            # Get LLM's next response (may request more tools)
            response = client.messages.create(max_tokens=max_tokens
                                              ,model=model
                                              ,tools=tools
                                              ,messages=messages)
        else:
            # No tools requested: conversation complete
            should_continue = False

## Chat Loop

In [31]:
def chat_loop():
    print("Type your queries or 'quit' to exit.")
    while True:
        try:
            query = input("\nQuery: ").strip()
            if query.lower() == 'quit':
                break

            process_query(query)
            print("\n")
        except Exception as e:
            print(f"\nError: {str(e)}")

In [35]:
chat_loop()

Type your queries or 'quit' to exit.

Query: LLM interpretability
I'll search for papers on LLM interpretability for you.
Calling tool search_papers with args {'topic': 'LLM interpretability', 'max_results': 5}
Results are saved in :papers/llm_interpretability/papers_info.json
Great! I found 5 papers on LLM interpretability. Let me extract detailed information about each of them.
Calling tool extract_info with args {'paper_id': '2407.04307v1'}
Calling tool extract_info with args {'paper_id': '2306.05212v1'}
Calling tool extract_info with args {'paper_id': '2509.03518v1'}
Calling tool extract_info with args {'paper_id': '2408.13006v2'}
Calling tool extract_info with args {'paper_id': '2407.07093v1'}
Here are the 5 papers I found on LLM interpretability:

## 1. **Crafting Large Language Models for Enhanced Interpretability** (2407.04307v1)
**Authors:** Chung-En Sun, Tuomas Oikarinen, Tsui-Wei Weng  
**Published:** July 5, 2024

This paper introduces **Concept Bottleneck Large Language Mo

In [36]:
chat_loop()

Type your queries or 'quit' to exit.

Query: LLM bias
I'll search for papers on LLM bias for you.
Calling tool search_papers with args {'topic': 'LLM bias', 'max_results': 5}
Results are saved in :papers/llm_bias/papers_info.json
Great! I found 5 papers on LLM bias. Let me extract detailed information about each of these papers.
Calling tool extract_info with args {'paper_id': '2404.11457v2'}
Calling tool extract_info with args {'paper_id': '2410.19775v1'}
Calling tool extract_info with args {'paper_id': '2109.14047v1'}
Calling tool extract_info with args {'paper_id': '2504.07887v2'}
Calling tool extract_info with args {'paper_id': '2407.04434v1'}
Here are 5 papers on LLM bias:

## 1. **Bias and Unfairness in Information Retrieval Systems: New Challenges in the LLM Era**
- **Authors**: Sunhao Dai, Chen Xu, Shicheng Xu, Liang Pang, Zhenhua Dong, Jun Xu
- **Published**: April 17, 2024
- **Summary**: A comprehensive survey examining emerging bias and unfairness issues in information retri

In [37]:
chat_loop()

Type your queries or 'quit' to exit.

Query: mutli modal retriever
I'll search for papers related to multi-modal retrievers for you.
Calling tool search_papers with args {'topic': 'multi modal retriever', 'max_results': 10}
Results are saved in :papers/multi_modal_retriever/papers_info.json
Great! I found several papers related to multi-modal retrievers. Let me extract more detailed information about these papers for you.
Calling tool extract_info with args {'paper_id': '2406.18007v1'}
Calling tool extract_info with args {'paper_id': '2407.19491v1'}
Calling tool extract_info with args {'paper_id': '2504.21375v1'}
Calling tool extract_info with args {'paper_id': '2311.04589v3'}
Calling tool extract_info with args {'paper_id': '2307.04751v1'}
Calling tool extract_info with args {'paper_id': '2403.17372v5'}
## Multi-Modal Retriever Papers Found

Here are the key papers on multi-modal retrieval and related topics:

### 1. **Deep Mamba Multi-modal Learning** (2406.18007v1)
   - **Authors:**